In [ ]:
%load_ext autoreload
%autoreload 2
import warnings

import os
import time
import pyproj
import numpy as np
import xarray as xr
import rioxarray
import s1_ard_core.computing.computation
import s1_ard_core.computing.coreg
import s1_ard_core.computing.dem
import s1_ard_core.computing.etad
import s1_ard_core.computing.gslc
import s1_ard_core.computing.io
import s1_ard_core.computing.orbit
import s1_ard_core.computing.shiroma
import s1_ard_core.computing.simul
from s1_ard_core.computing.io import read, write, normpath, Path
import s1_ard_core.computing.slc

import cop_ard_core.esd as esd_module
from s1_ard_core.computing.logger import logger
from pyproj.transformer import TransformerGroup

os.environ["PROJ_DATA"] = pyproj.datadir.get_data_dir()
warnings.filterwarnings("ignore")

In [ ]:
type, n_jobs, job_memory = "LOCAL", 4, "16GB"
client = s1_ard_core.computing.computation.create_client(type, n_jobs, job_memory)

In [ ]:
path = "/home/morea/SENTINEL1/TAIWAN/S1-RP"
slcs_path = f"{path}/slcs"
wdir_path_static = f"{path}/STATIC"
wdir_path_tmp = f"{path}/DYNAMIC"

orbit_path = None
reference_date = "2016-02-02"
chunks_sr = {"acquisition_date": 1, "polarization": 1, "azimuth_time": 2048, "slant_range_time": 2048}
chunks = {"x": 2048, "y": 2048}
chunk_length = max([v for v in chunks.values()])
coreg = True
gamma_flattening = True
recompute = False
multiburst = True

In [ ]:
slcs = read(slcs_path, consolidate_metadata=False)

In [ ]:
dem_tree = xr.DataTree()
for k, v in (slcs.bursts if multiburst else {"data": slcs}).items():
    path = f"bursts/{k}" if multiburst else "data"
    d = s1_ard_core.computing.io.read(f"{wdir_path_tmp}/{path}/dem_resampled", consolidate_metadata=False)
    if multiburst:
        dem_tree[path] = d
    else:
        dem_tree = xr.DataTree(d)

In [ ]:
orbits = s1_ard_core.computing.slc.extract_orbits(
    slcs,
    multiburst=multiburst,
    acquisition_date=[reference_date],
)
simulation_tree = s1_ard_core.computing.simul.stack_simulation(
    dem_tree=dem_tree,
    orbits=orbits,
    multiburst=multiburst,
    working_directory=wdir_path_static,
    chunks={**chunks, "acquisition_date": 1},
    master_image=True,
    recompute=recompute,
)
simulation_ml_tree = s1_ard_core.computing.simul.stack_simulation_ml(
    simulation_tree=simulation_tree,
    reference_date=reference_date,
    multiburst=multiburst,
    working_directory=wdir_path_static,
    recompute=recompute,
)
simulation_secondary_tree = xr.DataTree()
for k in simulation_tree.bursts if multiburst else {"data": simulation_tree}:
    path = f"bursts/{k}" if multiburst else "data"
    data_path = s1_ard_core.computing.io.normpath(f"{wdir_path_tmp}/{path}/simulation")
    if multiburst:
        simulation_secondary_tree[path] = s1_ard_core.computing.io.read(data_path, consolidate_metadata=False)
    else:
        simulation_secondary_tree = s1_ard_core.computing.io.read(data_path, consolidate_metadata=False)

In [ ]:
files = s1_ard_core.computing.io.glob(f"{wdir_path_tmp}/bursts/iw*")
for f in files:
    k = f.split("/")[-1]
    slcs[f"bursts/{k}/coreg"] = s1_ard_core.computing.io.read(f"{f}/coreg", consolidate_metadata=True)

In [ ]:
f_l = -10e6
f_h = 10e6
bw = 16e6

In [ ]:
y_poly_normal = esd_module.estimate(slcs, wdir_path_tmp, reference_date, poly_degree=2, enable_dask=True)

In [ ]:
y_poly_normal

In [ ]:
raise

In [ ]:
esd

In [ ]:
args = [
    esd.sel(acquisition_date=d, reference_date=rd)
    for d in esd.acquisition_date.values
    for rd in [reference_date]
    if d != rd
]

In [ ]:
esd_opt = s1_ard_core.computing.computation.map(
    esd_module.optimize_curvature, args=args, kwargs={"poly_degree": 0}, enable_dask=False
)

In [ ]:
esd_opt[0]

In [ ]:
esd_copy = esd.copy(deep=True)
for e in esd_opt:
    d, rd = e.acquisition_date.values, e.reference_date.values
    before = esd_copy["mean_phase"].loc[:, rd, d].copy(deep=True).values
    esd_copy["mean_phase"].loc[:, rd, d] = e.mean_phase
    after = esd_copy["mean_phase"].loc[:, rd, d].values
    print(after - before)
esd

In [ ]:
# y_poly_H = esd_module.estimate(slcs, wdir_path_tmp, reference_date, poly_degree=2, freq = f_h, bandwidth = bw,enable_dask=True)

In [ ]:
# y_poly_L["2016-02-14"].values, y_poly_normal["2016-02-14"].values, y_poly_H["2016-02-14"].values

In [ ]:
wdir_path_new = "/data/SENTINEL/TAIWAN/SUBBANDS"

In [ ]:
low_slcs = read(slcs_path, consolidate_metadata=False)

files = s1_ard_core.computing.io.glob(f"{wdir_path_new}/low/bursts/iw*")
for f in files:
    k = f.split("/")[-1]
    low_slcs[f"bursts/{k}/coreg/measurements"] = s1_ard_core.computing.io.read(
        f"{f}/coreg/measurements", consolidate_metadata=True
    )

In [ ]:
date = "2016-02-14"

In [ ]:
def get_interf_dt(cslc: xr.DataTree, multilook=dict, x_off: int = 0, y_off: int = 0):
    interf = (
        (cslc.isel(acquisition_date=1) * cslc.isel(acquisition_date=0).conj())
        .coarsen(multilook, boundary="trim")
        .mean()
    )
    a0 = np.sqrt((np.abs(cslc.isel(acquisition_date=0)) ** 2).coarsen(multilook, boundary="trim").mean())
    a1 = np.sqrt((np.abs(cslc.isel(acquisition_date=1)) ** 2).coarsen(multilook, boundary="trim").mean())
    interf = interf / (a0 * a1 + 1e-9)

    interf_dt = xr.Dataset()
    interf_dt["phase"] = xr.apply_ufunc(np.angle, interf, dask="parallelized")
    interf_dt["cohe"] = xr.apply_ufunc(np.abs, interf, dask="parallelized")
    interf_dt["azimuth_time"] = (
        multilook["azimuth_time"] / 2 - 0.5 + y_off + np.arange(interf_dt.azimuth_time.size) * multilook["azimuth_time"]
    )
    interf_dt["slant_range_time"] = (
        multilook["slant_range_time"] / 2
        - 0.5
        + x_off
        + np.arange(interf_dt.slant_range_time.size) * multilook["slant_range_time"]
    )
    return interf_dt

In [ ]:
def extract_interf(burst_id, slc, coreg, low_burst, bbox, dt_az_ml, dt_rg_ml, y_poly):
    print(f"processing burst {burst_id}")

    a_min, a_max, r_min, r_max = bbox

    if date not in slcs.bursts[burst_id].orig:
        return (burst_id, None)

    low_burst["daz"] = coreg.daz
    low_burst["drg"] = coreg.drg

    phase_L = cop_ard_core.utils.get_phramp_diff(low_burst, slc, y_poly, date)
    low_burst["cstack"].loc[date] *= np.exp(-1j * phase_L)
    cslc = low_burst.cstack

    # select area according to the output multilook grid
    print("Select area ...")
    x_min = np.ceil((cslc.slant_range_time.min().values - r_min) / dt_rg_ml) * dt_rg_ml + r_min
    x_min = np.round((x_min - cslc.slant_range_time.min().values) / dt_rg).astype(int)
    x_max = np.floor((cslc.slant_range_time.max().values - r_min) / dt_rg_ml) * dt_rg_ml + r_min
    x_max = np.round((x_max - cslc.slant_range_time.min().values) / dt_rg).astype(int)

    y_min = (np.ceil((cslc.azimuth_time.min().values - a_min).astype(float) / dt_az_ml) * dt_az_ml).astype(
        "timedelta64[ns]"
    ) + a_min
    y_min = np.round((y_min - cslc.azimuth_time.min().values).astype(float) / dt_az).astype(int)
    y_max = (np.floor((cslc.azimuth_time.max().values - a_min).astype(float) / dt_az_ml) * dt_az_ml).astype(
        "timedelta64[ns]"
    ) + a_min
    y_max = np.round((y_max - cslc.azimuth_time.min().values).astype(float) / dt_az).astype(int)
    cslc = cslc.isel(slant_range_time=slice(x_min, x_max), azimuth_time=slice(y_min, y_max))
    x_off = np.round((cslc.slant_range_time[0].values - r_min) / dt_rg).astype(int)
    y_off = np.round((cslc.azimuth_time[0].values - a_min).astype(float) / dt_az).astype(int)

    cslc["slant_range_time"] = np.arange(x_min + x_off, x_max + x_off)
    cslc["azimuth_time"] = np.arange(y_min + y_off, y_max + y_off) * dt_az

    # extract multilooked interferogram and coherence
    interf = get_interf_dt(cslc, multilook=multilook, x_off=x_off, y_off=y_off)
    interf["slant_range_time"] = interf.slant_range_time.values * dt_rg + r_min
    interf["azimuth_time"] = (interf.azimuth_time.values * dt_az).astype("timedelta64[ns]") + a_min
    interf = interf.compute().chunk(2048)
    return (burst_id, interf)

In [ ]:
import cop_ard_core.utils

interf_sub_band = xr.DataTree()
dt_az, dt_rg = cop_ard_core.utils.get_spacing(slcs.bursts, "coreg/measurements")
a_min, a_max, r_min, r_max = cop_ard_core.utils.get_bounding_box(slcs.bursts, "coreg/measurements")
multilook = {"azimuth_time": 16, "slant_range_time": 16 * 4}

dt_az_ml = dt_az * multilook["azimuth_time"]
dt_rg_ml = dt_rg * multilook["slant_range_time"]

In [ ]:
args = []
for i, (burst_id) in enumerate(slcs.bursts.keys()):
    if date not in slcs.bursts[burst_id].orig or "iw1" not in burst_id:
        continue
    args.append(
        (
            burst_id,
            slcs.bursts[burst_id].orig[date].measurements.slc,
            slcs.bursts[burst_id].coreg.measurements,
            low_slcs.bursts[burst_id].coreg.measurements.sel(polarization="V:V"),
        )
    )

In [ ]:
args = list(zip(*args))

In [ ]:
bbox = (a_min, a_max, r_min, r_max)

In [ ]:
results = s1_ard_core.computing.computation.map(
    extract_interf,
    args=args,
    kwargs={"bbox": bbox, "dt_az_ml": dt_az_ml, "dt_rg_ml": dt_rg_ml, "y_poly": y_poly_from_global_shift},
    enable_dask=True,
)

In [ ]:
interf_sub_band = xr.DataTree()
for burst_id, ds in results:
    interf_sub_band[f"bursts/{burst_id}"] = ds

In [ ]:
mosaic_L = (
    interf_sub_band.bursts[list(interf_sub_band.bursts.children)[0]]
    .to_dataset()
    .isel(azimuth_time=slice(0, -2), slant_range_time=slice(0, -5))
)
for k, layer in interf_sub_band.bursts.items():
    mosaic_L = mosaic_L.combine_first(layer.isel(azimuth_time=slice(0, -2), slant_range_time=slice(0, -5))).compute()

In [ ]:
write(mosaic_L, "/data/SENTINEL/TAIWAN/MOSAIC/mosaic_esd_full_band_from_global_shift_iw1", mode="w")

In [ ]:
# 0
mosaic_L.phase.plot.imshow()

In [ ]:
y_poly_L["2016-02-14"] - y_poly_normal["2016-02-14"], y_poly_normal["2016-02-14"]

In [ ]:
# -L
mosaic_L.phase.plot.imshow()

In [ ]:
y_poly_normal["2016-02-14"].values[:, 0], y_poly_L["2016-02-14"].values[:, 0]

In [ ]:
# +L
mosaic_L.phase.plot.imshow()

In [ ]:
ds = mosaic_L.phase
ds.drop_vars(ds.coords).rename({"azimuth_time": "y", "slant_range_time": "x"}).rio.to_raster(
    "/data/SENTINEL/TAIWAN/MOSAIC/mosaic_esd_full_band_from_global_shift_iw1.tiff", driver="COG"
)

In [ ]:
interf_comb_l = (
    interf_sub_band.bursts[list(interf_sub_band.bursts.children)[0]]
    .to_dataset()
    .isel(azimuth_time=slice(0, -2), slant_range_time=slice(0, -5))
)
for k, layer in interf_sub_band.bursts.items():
    interf_comb_l = interf_comb_l.combine_first(
        layer.isel(azimuth_time=slice(0, -2), slant_range_time=slice(0, -5))
    ).compute()

In [ ]:
interf_comb_l.phase.plot.imshow()

In [ ]:
interf_sub_band[list(interf_sub_band.children)[0]].to_dataset()

In [ ]:
import cop_ard_core.utils

for burst_id in slcs.bursts.keys():
    slc = slcs.bursts[burst_id].orig[date].measurements.slc
    coreg = slcs.bursts[burst_id].coreg.measurements
    burst = low_slcs.bursts[burst_id].coreg.measurements.sel(polarization="V:V")
    burst["daz"] = coreg.daz
    burst["drg"] = coreg.drg
    phase_L = cop_ard_core.utils.get_phramp_diff(burst, slc, y_poly_L, date)
    burst["cstack"].loc[date] *= np.exp(1j * phase_L)

In [ ]:
np.exp(1j * phase_L)

In [ ]:
y_poly[date]

In [ ]:
def get_coordinates(coreg, alpha, beta, coord_type: str = "pixels"):
    """
    Calculates coordinates for polynomial evaluation based on offsets and scaling.
    """
    y, x = coreg.azimuth_time, coreg.slant_range_time
    return (((e - alpha[i]).astype("float64") / beta[i].astype("float64")).values for i, e in enumerate([y, x]))


for burst, cslc in slcs.bursts.items():
    coreg = cslc.coreg.measurements
    dd = "2016-02-14"

    for y_poly in [y_poly_L, y_poly_normal, y_poly_H]:
        c1, c2 = get_coordinates(coreg, y_poly[dd].offset, y_poly[dd].scale_fact, y_poly.type)
        corr = np.polynomial.polynomial.polygrid2d(c1, c2, y_poly[dd].values)
        print("=" * 50)
        print(np.min(corr), np.max(corr))
    break

In [ ]:
slcs = s1_ard_core.computing.coreg.stack_coregistration(
    slcs,
    simulation_tree,
    simulation_ml_tree,
    reference_date,
    simulation_secondary_tree=simulation_secondary_tree,
    multiburst=multiburst,
    working_directory=f"{wdir_path_tmp}_ESD",
    y_poly=y_poly,
    recompute=True,
)

In [ ]:
normal_coreg = xr.DataTree()

files = s1_ard_core.computing.io.glob(f"{wdir_path_tmp}/bursts/iw*")
for f in files:
    k = f.split("/")[-1]
    normal_coreg[f"bursts/{k}/coreg"] = s1_ard_core.computing.io.read(f"{f}/coreg", consolidate_metadata=True)

In [ ]:
esd_coreg = xr.DataTree()

files = s1_ard_core.computing.io.glob(f"{wdir_path_tmp}_ESD/bursts/iw*")
for f in files:
    k = f.split("/")[-1]
    esd_coreg[f"bursts/{k}/coreg"] = s1_ard_core.computing.io.read(f"{f}/coreg", consolidate_metadata=True)

In [ ]:
cslc = slcs.bursts["iw1_146185"].coreg.measurements.cstack

In [ ]:
import xrft

frange = xrft.fft(
    cslc[1, 1].drop_vars(["slant_range_m", "azimuth_m"]).chunk(slant_range_time=-1),
    true_amplitude=True,
    true_phase=True,
    dim="slant_range_time",
).compute()
frange_2 = xrft.fft(
    cslc[1, 1]
    .isel(slant_range_time=slice(2048, 4096))
    .drop_vars(["slant_range_m", "azimuth_m"])
    .chunk(slant_range_time=-1),
    true_amplitude=True,
    true_phase=True,
    dim="slant_range_time",
).compute()

In [ ]:
p1 = (np.abs(frange) ** 2).mean(dim="azimuth_time")
p1 = p1 / np.max(p1)

p2 = (np.abs(frange_2) ** 2).mean(dim="azimuth_time")
p2 = p2 / np.max(p2)

In [ ]:
p1.plot()
p2.plot()

In [ ]:
frange_2